### Code for optimized two-step coarse propagators in parareal algorithm
#### Algorithm 4.1, optimize the two-step CP.


In [ ]:
import torch
import torch.nn as nn   
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam
from torch.autograd import Variable
from torch.optim.lr_scheduler import StepLR
from math import pi, cos, log
from scipy.special import factorial
from sympy import symbols, lambdify
from torch.utils.data import DataLoader, TensorDataset, Dataset
import numpy as np
from sympy import simplify
from scipy.sparse import diags
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim import LBFGS

In [ ]:
device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")  # Check for CUDA availability
N = 2000 
h = 1/N  
p2 = 1  
epochs = 500
K = 500 

# we start from BDF2.
b2 = torch.tensor([-1.],device=device) 
b1 = torch.tensor([log(2/3)],device=device)
a3_ = torch.tensor([0.],device=device)
a2_ = torch.tensor([0.005],device=device)
a1_ = torch.tensor([4/3],device=device)

d2 = torch.tensor([log(0.78849)],device=device)
d1 = torch.tensor([log(2/3)],device=device)
c3_ = torch.tensor([0.00],device=device)
c2_ = torch.tensor([0.00],device=device)
c1_ = torch.tensor([-1/3],device=device)

a1 = nn.Parameter(a1_, requires_grad=True)
a2 = nn.Parameter(a2_, requires_grad=True)
a3 = nn.Parameter(a3_, requires_grad=True)
b1_log = nn.Parameter(b1, requires_grad=True)
b2_log = nn.Parameter(b2, requires_grad=True)

c1 = nn.Parameter(c1_, requires_grad=True)
c2 = nn.Parameter(c2_, requires_grad=True)
c3 = nn.Parameter(c3_, requires_grad=True)
d1_log = nn.Parameter(d1, requires_grad=True)
d2_log = nn.Parameter(d2, requires_grad=True)


In [ ]:

def B01(x):
    return (a1 + a2 * x) / (1 + torch.exp(b1_log) * x)

def B_11(x):
    return ((1-a1) + c2 * x) / (1 + torch.exp(b1_log) * x)


def r(x):
    return torch.exp(-x)



def solve_alpha_beta(B0, B1):
    B0_complex = B0.type(torch.complex64)
    B1_complex = B1.type(torch.complex64)

    discriminant = torch.sqrt(B0_complex ** 2 + 4 * B1_complex)
    alpha = (B0_complex + discriminant) / 2
    beta = (B0_complex - discriminant) / 2

    return torch.abs(alpha), torch.abs(beta)





In [ ]:
lambdas1 = torch.tensor([-(2*cos(j*pi*h) - 2)/(h**2) for j in range(1, N)], dtype=torch.float32).to(device)
lambdas2 = torch.linspace(0.01, 100, 5000, device=device)
lambdas3 = torch.logspace(-4, 0, 5000, device=device)
lambdas = torch.cat((lambdas1, lambdas2, lambdas3))


dataset = torch.utils.data.TensorDataset(lambdas)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=len(dataset))

optimizer = torch.optim.SGD([a1,a2,b1_log,c2], lr=0.001)


scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, 
    step_size=K,     
    gamma=0.99        
)

# Tolerance for the gradient norm 
tolerance = 3e-1  

best_params = None
best_loss1 = float('inf')

stop_training = False

for epoch in range(epochs):
    p2 = p2 * 0.5
    for i in range(K):
        optimizer.zero_grad()
        
        alpha, beta = solve_alpha_beta(B01(lambdas), B_11(lambdas));
        B_10 =  (r(lambdas)- B01(lambdas)) * r(lambdas)  - B_11(lambdas);
        loss1 = torch.max(torch.abs(B_10)/(torch.abs((1 - torch.abs(alpha)) * (1- torch.abs(beta)))));
        loss2 = torch.mean(torch.log(1-torch.abs(alpha)**2) + torch.log(1-torch.abs(beta)**2))

        loss = loss1 - p2*(loss2)
        
        loss1.backward(retain_graph=True)
        
        grad_norm = 0.0
        for param in optimizer.param_groups[0]['params']:
            if param.grad is not None:
                grad_norm += param.grad.data.norm(2).item() ** 2
        grad_norm = 0.5 * grad_norm ** 0.5  

        if i % 100 == 0:
            print(f"Iteration {epoch}, epoch:{i}, Loss1: {loss1:.6f}, Loss2: {loss2:.6f}, Total Loss: {loss.item():.6f}")
        
        if loss1 < best_loss1:
            best_params = (a1.clone().detach(), a2.clone().detach(),a3.clone().detach(), b1_log.clone().detach(),b2_log.clone().detach(), c1.clone().detach(), c2.clone().detach(),c3.clone().detach())
            best_loss1 = loss1.item()
        

        if grad_norm < tolerance:
            print(f"Stopping criterion is fulfilled at epoch {epoch}, iteration {i}: gradient norm (w.r.t. loss1) {grad_norm:.2e} < {tolerance}")
            stop_training = True
            break 
        

        (-p2 * loss2).backward()
        
        optimizer.step()
        scheduler.step()

    if stop_training:
        break

print(f"Training stopped at epoch {epoch}, iteration {i} (gradient norm below tolerance)")

In [ ]:
best_loss1, best_params

In [ ]:
a1.data = best_params[0]
a2.data = best_params[1]
a3.data = best_params[2]
b1_log.data = best_params[3]
b2_log.data = best_params[4]

c1.data = 1 - a1.data
c2.data = best_params[6]
c3.data = best_params[7]

a = torch.cat((a1, a2), dim=0)
b = torch.cat((torch.ones(1, device=device), torch.exp(b1_log)),dim=0)
c = torch.cat((c1, c2), dim=0)

Px = " + ".join([f"{a[i].item():.5f}*x**{i}" for i in range(len(a))])
Qx = " + ".join([f"{b[i].item():.5f}*x**{i}" for i in range(len(b))])

P1x = " + ".join([f"{c[i].item():.5f}*x**{i}" for i in range(2)])

print(f"B_0 = ({Px}) / ({Qx})\nThe best convergence rate is {best_loss1:.6f}\n")
print(f"B_-1 = ({P1x}) / ({Qx})\nThe best convergence rate is {best_loss1:.6f}\n")